# Heterogeneous Effects (ROBUST)

The previous DML specifications estimate a single average effect. Here we let the
effect **vary across country-pairs** using a causal forest (`CausalForestDML`),
which estimates a conditional effect function θ(X) rather than one constant θ.
The forest partitions the space of pair characteristics X — economic size,
distance, contiguity, trade-agreement status — and estimates a separate sanction
effect in each region, discovering *data-drivenly* where sanctions bite hardest
rather than imposing a hypothesis a priori. As before, the outcome and treatment
are fixed-effects-residualized, so heterogeneity is identified from within-pair
variation. This answers a question the average effect cannot: not *whether*
sanctions reduce trade, but *for whom* the reduction is largest.

In the DML specifications above, random forests serve only as nuisance estimators — they residualize the outcome and treatment on confounders, while the effect itself remains a single constant estimated by a final linear step. **The causal forest differs fundamentally:** the forest is the effect estimator, splitting on covariates to recover a conditional effect function θ(X) rather than one average number.

**Causal Forests: how they differ from standard random forests**

**Standard RF** predicts an outcome: each tree splits the feature space to minimize
prediction error, and the forest averages predicted $\hat{y}(x)$ across trees. The
target is $\mathbb{E}[Y \mid X=x]$.

**Causal forests** estimate a *treatment effect*, not an outcome. The target is the
conditional average treatment effect (CATE):

$$
\tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]
$$

Three key differences:

**1. Splitting criterion.** A standard tree splits to make $Y$ more homogeneous within
leaves. A causal tree splits to make the *treatment effect* $\tau$ more heterogeneous
*across* leaves — it looks for regions of $X$ where the effect differs, not where the
outcome differs.

**2. Orthogonalization (DML residualization).** Within `CausalForestDML`, the outcome
and treatment are first residualized on the controls,
$\tilde{Y} = Y - \hat{m}(W)$ and $\tilde{D} = D - \hat{e}(W)$, so the forest estimates
$\tau(x)$ from the confounder-free residual relationship. This is what makes the CATE
causal rather than merely predictive.

**3. Honesty.** Each tree uses one subsample to decide *where* to split and a separate
subsample to *estimate* the effect in each leaf. This "honest" split prevents the same
data from both finding and measuring an effect, which would overstate heterogeneity —
and it enables valid confidence intervals for $\tau(x)$.

In short: a standard forest asks *"what is $Y$ here?"*; a causal forest asks
*"how much does the treatment change $Y$ here?"* — and is built so that answer is
identified and honestly measured.

#### Libraries

In [ ]:
!pip install -q econml pyfixest
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf
from sklearn.ensemble import RandomForestRegressor
from econml.dml import CausalForestDML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 912.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 33.5 MB/s eta 0:00:00


In [ ]:
SEED = 123

Connect to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'
export_path = '/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/'

Data uploading

In [ ]:
# ── load + feature lists ──
merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",
                            dtype={"exp_iso3": str, "imp_iso3": str})
merged_main = merged_robust   # alias — downstream cells unchanged
econ  = ["exp_gdp","imp_gdp","exp_pop","imp_pop","dist_w_harm"]
flags = ["contig","comlang","comcol","colony","fta","exp_eu","imp_eu","exp_wto","imp_wto"]
print(merged_main.shape)

/tmp/ipykernel_2082/1433892486.py:2: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_robust = pd.read_csv(import_path + "merged_robust_final.csv",


(1404170, 54)


### FE residualization


In [ ]:
# ── FE residualization (y_res, D_res) + aligned rows ──
fe = merged_main[["exp_iso3","imp_iso3","year"]].copy()
fe["y"]    = np.log1p(merged_main["trade"].values)
fe["D"]    = merged_main["sanctioned_any"].values.astype(float)
fe["ey"]   = fe["exp_iso3"] + "_" + fe["year"].astype(str)
fe["iy"]   = fe["imp_iso3"] + "_" + fe["year"].astype(str)
fe["pair"] = fe["exp_iso3"] + "_" + fe["imp_iso3"]

y_res = np.asarray(pf.feols("y ~ 1 | ey + iy + pair", data=fe).resid())
D_res = np.asarray(pf.feols("D ~ 1 | ey + iy + pair", data=fe).resid())

keep = ((fe.groupby("ey")["ey"].transform("size") > 1) &
        (fe.groupby("iy")["iy"].transform("size") > 1) &
        (fe.groupby("pair")["pair"].transform("size") > 1))
mm = merged_main[keep.values].reset_index(drop=True)
print("aligned:", len(y_res), len(D_res), len(mm))

/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 6 singleton fixed effect(s) dropped from the model.
  warnings.warn(


aligned: 1404164 1404164 1404164


In [ ]:
# ── build X (effect modifiers), W (controls) — full sample ──
from sklearn.impute import SimpleImputer

X_cols = ["exp_gdp","imp_gdp","dist_w_harm","contig","fta"]     # what the effect varies with
W_cols = econ + flags                                           # nuisance controls

Xdf = mm[X_cols].copy()
for c in ["exp_gdp","imp_gdp","dist_w_harm"]:                   # log the skewed ones
    Xdf[c] = np.log1p(Xdf[c])
Xdf = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(Xdf), columns=X_cols)

Wdf = mm[W_cols].copy()
for c in econ: Wdf[c] = np.log1p(Wdf[c])
Wdf[econ]  = SimpleImputer(strategy="median").fit_transform(Wdf[econ])
Wdf[flags] = Wdf[flags].fillna(0)

Xs, Ws = Xdf.values, Wdf.values      # full sample — no [idx]
ys, Ds = y_res, D_res
print("full sample:", ys.shape, "| X:", Xs.shape, "| W:", Ws.shape)

full sample: (1404164,) | X: (1404164, 5) | W: (1404164, 14)


## Causal Forest DML on FE, FE-aug.

In [ ]:
# ── Causal Forest DML on FE-residualized y, D ──
cf = CausalForestDML(
    model_y = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    model_t = RandomForestRegressor(n_estimators=100, min_samples_leaf=50,
                                    max_samples=0.5, random_state=SEED, n_jobs=-1),
    n_estimators=200, min_samples_leaf=50, max_samples=0.4,
    cv=3, random_state=SEED
)
cf.fit(ys, Ds, X=Xs, W=Ws)

ate = cf.ate(Xs); ate_ci = cf.ate_interval(Xs)          # average effect (sanity vs  ~-0.05)
print(f"ATE: {ate:.4f}  CI [{ate_ci[0]:.4f}, {ate_ci[1]:.4f}]")

ATE: -0.0816  CI [-2.0242, 1.8610]


In [ ]:
import joblib
joblib.dump(cf, import_path + "causal_forest_rob.pkl")

['/content/drive/MyDrive/IMT_studies/ML_econ/Project/Data/Cleaned/causal_forest_rob.pkl']

## Any signs of hetero.?

In [ ]:
# ── is there heterogeneity worth interpreting? ──
te = cf.effect(Xs)                          # per-observation effect estimates θ(X)

print(f"ATE (mean effect): {te.mean():.4f}")
print(f"SD of effects:     {te.std():.4f}")
print(f"range:             [{te.min():.4f}, {te.max():.4f}]")
print(f"10th–90th pct:     [{np.percentile(te,10):.4f}, {np.percentile(te,90):.4f}]")

ATE (mean effect): -0.0816
SD of effects:     0.9996
range:             [-27.6025, 32.9992]
10th–90th pct:     [-0.9855, 0.7909]


In [ ]:
# is the heterogeneity statistically real, or just noise?
# test whether effect variation exceeds what noise would produce
import numpy as np
te_inf = cf.effect_inference(Xs)
summ = te_inf.summary_frame()
sig = (summ["pvalue"] < 0.05).mean()
print(f"share of pairs with individually significant effect: {sig:.1%}")

# compare terciles of the strongest modifier (once we know it)
print(cf.feature_importances_)          # which X drives splits

share of pairs with individually significant effect: 11.0%
[0.11018783 0.32378378 0.48061853 0.00331946 0.08209041]


A causal forest (CausalForestDML) estimated on the fixed-effects-residualized data reveals no statistically robust heterogeneity in the aggregate sanction effect. The share of pairs with an individually significant effect (10.4%) is close to what noise alone would produce, feature importances are unstable across sample sizes, and the distribution of conditional effects is dominated by estimation variance (implausibly wide once individual). We conclude the average effect is an adequate summary: the sanction impact does not vary systematically with pair economic size, distance, or trade-agreement status in a way the data can reliably identify. This is consistent with the limited within-pair variation in sanction status, which supports estimation of an average effect but not of finely conditional ones.